Viewed googlenet.ipynb:1-400
Viewed googlenet.ipynb:1200-1361

以下是对 [googlenet.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_convolutional-modern/googlenet.ipynb) 内容的系统总结：

---

### 一、核心设计背景与思想

在 GoogLeNet（Inception-v1，2014 年 ImageNet 冠军）提出之前，主流网络（如 AlexNet、VGG）面临两个核心问题：
1. **卷积核尺度的选择困境**：不同尺寸的卷积核各有优劣（小卷积核提取局部细节，大卷积核感受野更广），过去难以权衡哪种最佳。
2. **模型参数爆炸与计算代价高**：传统的深层网络往往包含巨型全连接层，参数密集且容易过拟合。

**GoogLeNet 的核心思想**：
* **多尺度并行特征提取**：不再强求选择单一核大小，而是通过不同尺度的卷积并行处理，最后按通道拼接。
* **1×1 卷积降维（BottleNeck）**：借鉴 NiN 思想，在计算量大的卷积前使用 `1×1` 卷积降低通道数，大幅压缩计算量与参数量。
* **全局平均池化取代全连接层**：网络尾部使用全局平均池化，直接消除数以千万计的全连接层参数。

---

### 二、Inception 块结构拆解

Inception 块是 GoogLeNet 的基础单元，包含 **4 条并行分支**：

```text
                  输入 (in_channels)
         ┌────────────┬─────────────┬─────────────┐
         │            │             │             │
      1×1 卷积     1×1 卷积      1×1 卷积      3×3 最大池化
      (通道 c1)   (降维 c2[0])   (降维 c3[0])   (stride=1, pad=1)
         │            │             │             │
         │         3×3 卷积      5×5 卷积      1×1 卷积
         │        (通道 c2[1])   (通道 c3[1])   (通道 c4)
         │            │             │             │
         └────────────┴─────────────┴─────────────┘
                             │
                      通道拼接 (cat, dim=1)
                             │
                    输出通道 = c1 + c2[1] + c3[1] + c4
```

* **高宽保持一致**：通过设置相应的填充（3×3 卷积 pad=1，5×5 卷积 pad=2，3×3 池化 pad=1），确保 4 个分支输出的特征图尺寸完全一致。
* **通道融合**：使用 `torch.cat((p1, p2, p3, p4), dim=1)` 在通道维度进行级联。

---

### 三、GoogLeNet 网络架构（5 个模块）

GoogLeNet 共堆叠了 **9 个 Inception 块**，整个网络可划分为 5 个阶段模块：

| 模块 | 主要结构与操作 | 特征图形状变化 (输入 1×96×96) | 说明 |
| :--- | :--- | :--- | :--- |
| **b1** | 7×7 卷积 (stride=2, pad=3) + ReLU + 3×3 MaxPool (stride=2) | `(1, 64, 24, 24)` | 前端快速下采样，提取低阶图像边缘特征 |
| **b2** | 1×1 卷积 (64通道) + 3×3 卷积 (192通道) + 3×3 MaxPool (stride=2) | `(1, 192, 12, 12)` | 降维后再升维，对应 Inception 支路2的结构 |
| **b3** | 串联 2 个 Inception 块 (输出通道 256、480) + 3×3 MaxPool (stride=2) | `(1, 480, 6, 6)` | 通道数逐步扩大，降采样后送入深层 |
| **b4** | 串联 5 个 Inception 块 (输出通道 512, 512, 512, 528, 832) + 3×3 MaxPool (stride=2) | `(1, 832, 3, 3)` | 网络主体层，抽取丰富的深层语义特征 |
| **b5** | 串联 2 个 Inception 块 (输出通道 832, 1024) + `AdaptiveAvgPool2d((1,1))` + `Flatten()` | `(1, 1024)` | 全局平均池化将高宽压为 1×1，展平成 1024 维向量 |
| **输出层** | `nn.Linear(1024, 10)` | `(1, 10)` | 分类头映射为类别概率输出 |

---

### 四、训练与实验

1. **输入分辨率调整**：
   * 原版 ImageNet 输入分辨率为 `224 × 224`；
   * 教程中为了适配 Fashion-MNIST 并加速训练，将图片双线性插值缩放为 `96 × 96`（刚好能满足多次下采样后的最小空间尺寸要求）。
2. **训练超参数**：
   * 学习率 `lr = 0.1`，轮数 `num_epochs = 10`，批大小 `batch_size = 128`。
   * 使用 `d2l.train_ch6` 在 GPU 上进行多轮端到端训练与验证。

---

### 五、要点总结与演进方向

* **计算效率极高**：相比于参数量动辄上亿的 AlexNet 与 VGG，GoogLeNet 仅有几百万参数，却拥有更出色的识别准确率。
* **设计权衡**：Inception 各支路的通道比例并非手工随意设定，而是通过大量实验探索得到的经验配比。
* **后续经典演进**：
  * **Inception-v2**：引入批量规范化（Batch Normalization），将 5×5 卷积替换为两个连续的 3×3 卷积。
  * **Inception-v3**：引入不对称卷积分解（如 1×7 与 7×1）进一步削减参数。
  * **Inception-v4 / Inception-ResNet**：结合残差连接（Residual Connection），提升极深网络的训练稳定性。

In [2]:
import torch 
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [3]:
class Inception(nn.Module):

    def __init__(self, in_channels, c1, c2, c3, c4, **kwargs):
        super(Inception, self).__init__(**kwargs)

        self.p1_1 = nn.Conv2d(in_channels, c1, kernel_size=1)

        self.p2_1 = nn.Conv2d(in_channels, c2[0], kernel_size=1)
        self.p2_2 = nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1)


        self.p3_1 = nn.Conv2d(in_channels, c3[0], kernel_size=1)
        self.p3_2 = nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2)

        self.p4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.p4_2 = nn.Conv2d(in_channels, c4, kernel_size=1)
    
    def forward(self, X):
        p1 = F.relu(self.p1_1(X))
        p2 = F.relu(self.p2_2(F.relu(self.p2_1(X))))
        p3 = F.relu(self.p3_2(F.relu(self.p3_1(X))))
        p4 = F.relu(self.p4_2(self.p4_1(X)))
        return torch.cat((p1, p2, p3, p4), dim=1)


In [4]:
b1 = nn.Sequential(
    nn.Conv2d(1, 64, kernel_size=7, padding=3, stride=2),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

In [5]:
b2 = nn.Sequential(
    nn.Conv2d(64, 64, kernel_size=1),
    nn.ReLU(),
    nn.Conv2d(64, 192, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

In [6]:
b3 = nn.Sequential(
    Inception(192, 64, (96, 128), (16, 32), 32),
    Inception(256, 128, (128, 192), (32, 96), 64),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

In [7]:
b4 = nn.Sequential(Inception(480, 192, (96, 208), (16, 48), 64),
                   Inception(512, 160, (112, 224), (24, 64), 64),
                   Inception(512, 128, (128, 256), (24, 64), 64),
                   Inception(512, 112, (144, 288), (32, 64), 64),
                   Inception(528, 256, (160, 320), (32, 128), 128),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

In [8]:
b5 = nn.Sequential(
    Inception(832, 256, (160, 320), (32, 128), 128),
    Inception(832, 384, (192, 384), (48, 128), 128),
    nn.AdaptiveAvgPool2d((1,1)),
    nn.Flatten()
)

In [9]:
net = nn.Sequential(b1, b2, b3, b4, b5, nn.Linear(1024, 10))

In [10]:
X = torch.rand(size=(1, 1, 96, 96))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 64, 24, 24])
Sequential output shape:	 torch.Size([1, 192, 12, 12])
Sequential output shape:	 torch.Size([1, 480, 6, 6])
Sequential output shape:	 torch.Size([1, 832, 3, 3])
Sequential output shape:	 torch.Size([1, 1024])
Linear output shape:	 torch.Size([1, 10])


In [ ]:
lr, num_epochs, batch_size = 0.1, 10, 128
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=96)
d2l.train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())

以下是针对 [googlenet.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_convolutional-modern/googlenet.ipynb) 课后练习题的详细解答与分析：

---

### 练习 1：GoogLeNet 后续版本的关键技术演进

GoogLeNet 在提出后经历了一系列经典的架构迭代（Inception-v2 到 v4，以及 Inception-ResNet）。这些版本的核心改进包括：

#### 1. 批量规范化（Batch Normalization, Inception-v2）
* **原理**：在每个卷积层之后、激活函数之前加入 BN 层（BatchNorm2d）。
* **作用**：
  * 缓解深层网络中的内部协变量偏移（Internal Covariate Shift）。
  * 允许使用更高的初始学习率（如将原版的 0.01/0.1 提高数倍），大幅提升训练收敛速度。
  * 具有轻微的正则化效应，降低对 Dropout 的依赖。

#### 2. Inception 模块结构调整（卷积核分解，Inception-v3）
* **以小换大（两层 3×3 替换 5×5）**：
  * 感受野分析：两个连续的 `3×3` 卷积等效于一个 `5×5` 卷积。
  * 参数量对比：参数由 `5×5 = 25` 降为 `2 × (3×3) = 18`，参数量与计算量减少约 28%，同时多引入了一次非线性激活层。
* **非对称卷积分解（1×n 与 n×1）**：
  * 将 `n×n` 卷积进一步拆解为串联的 `1×n` 和 `n×1` 卷积（例如 `1×3` 接 `3×1`，或中深层的 `1×7` 接 `7×1`）。
  * 在保持感受野的前提下，计算量进一步降至原来的 `2/n`，使网络可以做得更深更宽。

#### 3. 标签平滑正则化（Label Smoothing Regularization）
* **传统 One-Hot 标签的问题**：真实类别概率为 1，其他为 0。交叉熵损失驱动分类层输出极大的 logits，容易导致模型过度自信（overfitting）。
* **平滑标签构造**：设类别总数为 `K`，平滑因子为 `ε`（如 0.1）：
  ```text
  q(y) = 1 - ε         （当 y 为真实标签）
  q(y) = ε / K         （当 y 为其他负标签）
  ```
* **效果**：软化了目标分布，防止模型对训练样本输出过度极端的置信度，显著提升了测试泛化能力。

#### 4. 加入残差连接（Inception-ResNet-v1 / v2）
* **机制**：在 Inception 模块外部增加跨层直连通道（Identity Shortcut）：
  ```text
  y = ReLU(Inception(x) · W_linear + x)
  ```
* **效果**：融合了 ResNet 的残差传播优势与 Inception 的多尺度提取优势，有效解决了梯度消失问题，极大加速了超深网络的训练收敛。

---

### 练习 2：GoogLeNet 的最小图像输入尺寸是多少？

#### 尺寸下采样推导
GoogLeNet 在输入到达尾部的全局平均汇聚层之前，共经历了 **5 次步长为 2 的空间下采样（减半）**：

1. **b1 阶段（2 次）**：
   * `Conv2d(7×7, stride=2, padding=3)`：空间尺寸减半；
   * `MaxPool2d(3×3, stride=2, padding=1)`：空间尺寸减半。
2. **b2 阶段（1 次）**：
   * `MaxPool2d(3×3, stride=2, padding=1)`：空间尺寸减半。
3. **b3 阶段（1 次）**：
   * `MaxPool2d(3×3, stride=2, padding=1)`：空间尺寸减半。
4. **b4 阶段（1 次）**：
   * `MaxPool2d(3×3, stride=2, padding=1)`：空间尺寸减半。
5. **b5 阶段（0 次下采样，仅保持尺寸）**：
   * 两个 Inception 块内部高宽保持不变；
   * 尾部接 `AdaptiveAvgPool2d((1, 1))`。

总下采样倍数为：
```text
总下采样因子 = 2 × 2 × 2 × 2 × 2 = 2⁵ = 32
```

#### 结论
* **理论最小尺寸：`32 × 32`**
  * 若输入为 `32 × 32`，经过 5 次步长为 2 的下采样后，到达 b5 时的特征图空间尺寸恰好缩减为 `1 × 1`，全局平均池化和全连接分类层仍可正常执行。
  * 若输入小于 `32 × 32`（如 16×16），特征图在 b4 阶段就会提前退化为 1×1 甚至 0，后续的池化层与 5×5 卷积（pad=2）将失去多尺度特征提取的几何意义。
* **工程推荐尺寸**：
  * 虽然 `32 × 32` 在计算上合法，但由于前端存在 `7×7` 的大卷积核，且中后段存在大量 `3×3` 与 `5×5` 卷积，若输入仅为 32×32，特征图分辨率衰减过快；
  * 因此实际应用中，如 notebook 中展示的 **`96 × 96`** 或原论文中的 **`224 × 224`** 是能充分发挥 GoogLeNet 多尺度表征能力的合理尺寸。

---

### 练习 3：AlexNet、VGG、NiN 与 GoogLeNet 的参数量对比及压缩秘诀

#### 1. 经典架构参数量级对比

| 模型名称 | 大致总参数量 | Top-5 错误率 (ImageNet) | 主要参数集中区域 |
| :--- | :--- | :--- | :--- |
| **AlexNet** | 约 60M (6000 万) | 约 16.4% | 全连接层（FC6 单层占约 37.7M，超过 60%） |
| **VGG-16** | 约 138M (1.38 亿) | 约 7.3% | 全连接层（FC6 占约 102M，全连接层占 80%+） |
| **NiN** | 约 2M (200 万) | - (主要用于探索) | 各层 1×1 卷积（完全消除全连接层） |
| **GoogLeNet** | 约 5M ~ 7M (约 600 万) | 约 6.7% | 卷积层（Inception 块内） |

#### 2. NiN 与 GoogLeNet 显著减少参数量的核心机制

后两个网络相比 AlexNet 和 VGG，参数量下降了 **一个数量级以上（减少 90%~95%）**，核心归功于以下两项革命性设计：

##### （1）全面摒弃巨型全连接层，引入「全局平均汇聚（Global Average Pooling, GAP）」
* **旧模式（AlexNet / VGG）**：
  * 在卷积特征提取完毕后，将例如 `512 × 7 × 7 = 25088` 维的空间特征直接摊平成向量，送入 4096 维全连接层。
  * 单是这一层权重矩阵的参数量就是：
    ```text
    参数量 = 25088 × 4096 ≈ 1.02 亿参数！
    ```
* **新模式（NiN / GoogLeNet）**：
  * **NiN**：最后一个卷积层直接将通道数输出为分类类别数（如 10 类），然后通过全局平均池化将每个通道的 `H × W` 求均值变成一个标量，直接作为分类 Logits，**参数量为 0**。
  * **GoogLeNet**：在 b5 阶段末尾使用 `AdaptiveAvgPool2d((1, 1))`，将特征图从 `(1024, H, W)` 压缩为 `(1024, 1, 1)`，后续仅接一个极轻量的分类头 `Linear(1024, num_classes)`。对于 1000 分类而言，该层仅需要 `1024 × 1000 ≈ 1M` 的参数，彻底消除了数千万至上亿的冗余权重。

##### （2）Inception 块内部通过 1×1 卷积实施「降维瓶颈（BottleNeck）」
* 在 GoogLeNet 的 Inception 块中，计算昂贵的 `3×3` 和 `5×5` 卷积**并不直接处理高维度输入**，而是先由 `1×1` 卷积将通道数压缩到较低水平（如 1/2 或 1/4），再进行较大尺度的卷积运算。
* **参数量节省计算示例**：
  假设输入特征图通道为 256，需要输出 128 个通道的 5×5 特征：
  * **直接使用 5×5 卷积**：
    ```text
    直接参数量 = 256 × 128 × 5 × 5 = 819,200
    ```
  * **先用 1×1 降维至 32，再接 5×5 卷积（GoogLeNet 方式）**：
    ```text
    1×1 卷积参数量 = 256 × 32 × 1 × 1 = 8,192
    5×5 卷积参数量 = 32 × 128 × 5 × 5 = 102,400
    总参数量 = 8,192 + 102,400 = 110,592  (节省了约 86.5% 的参数与计算量！)
    ```